<a href="https://colab.research.google.com/github/MEL313/MEL313/blob/main/Copy_of_Image_classifier_object_Isolation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install fiftyone


In [ ]:
# prompt: install tensorflow.examples
!pip install -q git+https://github.com/tensorflow/examples.git


In [ ]:
import os
import glob
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import tensorflow as tf, keras
import numpy as np
import random
import os.path as path
import tensorflow_datasets as tfds
import collections
import concurrent.futures
import re

from etils import epath
import numpy as np
from tensorflow_datasets.core.utils.lazy_imports_utils import tensorflow as tf
import tensorflow_datasets.public_api as tfds

from scipy import misc
from keras.preprocessing import image
from tensorflow.keras.preprocessing.image import ImageDataGenerator, img_to_array, load_img
from google.colab import files

from IPython.display import clear_output

In [ ]:
!wget https://drive.google.com/file/d/1UT1vhV6pPzZo-LGXKit8Q3GheFVf5RkC/view?usp=sharing

In [ ]:
!ls


In [ ]:
import fiftyone as fo
import fiftyone.zoo as foz
from google.colab import drive

# Mount Google Drive
drive.mount('/content/drive')

# A name for the dataset
name = "airplane_test"

# The type of data being imported
dataset_type = fo.types.COCODetectionDataset
!unzip -j "/content/drive/MyDrive/planes.zip" -d "/content/planes"

dataset = fo.Dataset.from_dir(
    dataset_dir=r"/content/planes",
    data_path=".",  # Images are directly in /content/planes
    labels_path=r"bbox.json", # labels.json is directly in /content/planes
    dataset_type=fo.types.COCODetectionDataset
)

session = fo.launch_app(dataset, port=5151)
session.wait()

In [ ]:
!unzip -q "/content/drive/MyDrive/planes.zip" -d "/content/planes" # Make sure files are unzipped and -q suppresses verbose output

# Verify image files exist in the directory
!ls "/content/planes"

In [ ]:
# prompt: load the planes images from planes file

planes_images = tf.keras.utils.image_dataset_from_directory(
    '/content/planes',
    image_size=(128, 128),
    batch_size=64)


In [ ]:
def normalize(input_image, input_mask):
  input_image = tf.cast(input_image, tf.float32) / 255.0
  input_mask -= 1
  return input_image, input_mask

In [ ]:
def load_image(input_image, label): # Add label as an argument
  input_image = tf.image.resize(input_image, (256, 256))

  input_image, _ = normalize(input_image, 0) # Pass a dummy value for input_mask

  return input_image, label # Return the label as well

In [ ]:
# split the training and test data

train_size = int(0.7 * len(planes_images))
test_size = len(planes_images) - train_size
train_dataset = planes_images.take(train_size)
test_dataset = planes_images.skip(train_size)



In [ ]:
# Define info as a dictionary with an empty 'splits' key
info = {'splits': {}}

# Add the training and test datasets to the 'splits' dictionary (replace with actual data)
info['splits']['train'] = train_dataset
info['splits']['test'] = test_dataset

TRAIN_LENGTH = info['splits']['train'].cardinality().numpy()  # Use cardinality to get number of elements
BATCH_SIZE = 64
BUFFER_SIZE = 1000
STEPS_PER_EPOCH = TRAIN_LENGTH // BATCH_SIZE

In [ ]:
# prompt: set autotune to work with training and test data not using map

train_dataset = train_dataset.map(load_image, num_parallel_calls=tf.data.AUTOTUNE)
test_dataset = test_dataset.map(load_image, num_parallel_calls=tf.data.AUTOTUNE)


In [ ]:
# Define info as an empty object
info = {planes_images}


In [ ]:
class Augment(tf.keras.layers.Layer):
  def __init__(self, seed=42):
    super().__init__()
    # both use the same seed, so they'll make the same random changes.
    self.augment_inputs = tf.keras.layers.RandomFlip(mode="horizontal", seed=seed)


  def call(self, inputs, labels):
    inputs = self.augment_inputs(inputs)

    return inputs, labels


In [ ]:
train_batches = (
    train_dataset
    .cache()
    .shuffle(BUFFER_SIZE)
    # Remove the batching here
    #.batch(BATCH_SIZE)
    .repeat()
    .map(Augment())
    .batch(BATCH_SIZE) # Batch after the Augment transformation
    .prefetch(buffer_size=tf.data.AUTOTUNE))

test_batches = test_dataset.batch(BATCH_SIZE)

In [ ]:
def display(display_list):
  plt.figure(figsize=(15, 15))

  title = ['Input Image', 'True Mask', 'Predicted Mask']

  for i in range(len(display_list)):
    plt.subplot(1, len(display_list), i+1)
    plt.title(title[i])
    plt.imshow(tf.keras.utils.array_to_img(display_list[i]))
    plt.axis('off')
  plt.show()


In [ ]:
for images, masks in train_batches.take(2):
  # Iterate over images in the batch
  for i in range(images.shape[0]):
    sample_image, sample_mask = images[i], masks[i]
    display([sample_image, sample_mask])


In [ ]:
base_model = tf.keras.applications.MobileNetV2(input_shape=[128, 128, 3], include_top=False)

# Use the activations of these layers
layer_names = [
    'block_1_expand_relu',   # 64x64
    'block_3_expand_relu',   # 32x32
    'block_6_expand_relu',   # 16x16
    'block_13_expand_relu',  # 8x8
    'block_16_project',      # 4x4
]
base_model_outputs = [base_model.get_layer(name).output for name in layer_names]

# Create the feature extraction model
down_stack = tf.keras.Model(inputs=base_model.input, outputs=base_model_outputs)

down_stack.trainable = False

In [ ]:
# @title
from tensorflow_examples.models.pix2pix import pix2pix

up_stack = [
    pix2pix.upsample(512, 3),  # 4x4 -> 8x8
    pix2pix.upsample(256, 3),  # 8x8 -> 16x16
    pix2pix.upsample(128, 3),  # 16x16 -> 32x32
    pix2pix.upsample(64, 3),   # 32x32 -> 64x64
]

In [ ]:
def unet_model(output_channels:int):
  inputs = tf.keras.layers.Input(shape=[128, 128, 3])

  # Downsampling through the model
  skips = down_stack(inputs)
  x = skips[-1]
  skips = reversed(skips[:-1])

  # Upsampling and establishing the skip connections
  for up, skip in zip(up_stack, skips):
    x = up(x)
    concat = tf.keras.layers.Concatenate()
    x = concat([x, skip])

  # This is the last layer of the model
  last = tf.keras.layers.Conv2DTranspose(
      filters=output_channels, kernel_size=3, strides=2,
      padding='same')  #64x64 -> 128x128

  x = last(x)

  return tf.keras.Model(inputs=inputs, outputs=x)

In [ ]:
OUTPUT_CLASSES = 3

model = unet_model(output_channels=OUTPUT_CLASSES)
model.compile(optimizer='adam',
              loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True),
              metrics=['accuracy'])

In [ ]:
tf.keras.utils.plot_model(model, show_shapes=True, expand_nested=True, dpi=64)


In [ ]:
def create_mask(pred_mask):
  pred_mask = tf.math.argmax(pred_mask, axis=-1)
  pred_mask = pred_mask[..., tf.newaxis]
  return pred_mask[0]

In [ ]:
def show_predictions(dataset=None, num=1):
  if dataset:
    for image, mask in dataset.take(num):
      pred_mask = model.predict(image)
      display([image[0], mask[0], create_mask(pred_mask)])
  else:
    display([sample_image, sample_mask,
             create_mask(model.predict(sample_image[tf.newaxis, ...]))])

In [ ]:
class DisplayCallback(tf.keras.callbacks.Callback):
  def on_epoch_end(self, epoch, logs=None):
    clear_output(wait=True)
    show_predictions()
    print ('\nSample Prediction after epoch {}\n'.format(epoch+1))

In [ ]:
# prompt: establish callback function

class myCallback(tf.keras.callbacks.Callback):
  def on_epoch_end(self, epoch, logs={}):
    if(logs.get('accuracy')>0.90):
      print("\nReached 90% accuracy so cancelling training!")
      self.model.stop_training = True

callbacks = myCallback()


In [ ]:
# prompt: define num_train_samples

TRAIN_LENGTH = info['splits']['train'].cardinality().numpy()


In [ ]:
EPOCHS = 20
VAL_SUBSPLITS = 5

# Assuming you have a TensorFlow dataset and its info object
# Replace `info` with the correct way to access your dataset information
dataset, info = tfds.load('your_dataset_name', with_info=True)

# Correctly access the number of examples in the test split
num_test_examples = info.splits['test'].num_examples
VALIDATION_STEPS = num_test_examples // BATCH_SIZE // VAL_SUBSPLITS

model_history = model.fit(train_batches, epochs=EPOCHS,
                          steps_per_epoch=STEPS_PER_EPOCH,
                          validation_steps=VALIDATION_STEPS,
                          validation_data=test_batches,
                          callbacks=[DisplayCallback()])


In [ ]:
EPOCHS = 20
VAL_SUBSPLITS = 5



# Correctly access the number of examples in the test split
num_test_examples = info.splits['test'].num_examples
VALIDATION_STEPS = num_test_examples // BATCH_SIZE // VAL_SUBSPLITS

model_history = model.fit(train_batches, epochs=EPOCHS,
                          steps_per_epoch=STEPS_PER_EPOCH,
                          validation_steps=VALIDATION_STEPS,
                          validation_data=test_batches,
                          callbacks=[DisplayCallback()])


In [ ]:
show_predictions(test_batches, num=3)